# S6E10 — 06 MLP (a neural network built step by step)

> **Pipeline position:** 05 CatBoost/XGB → **06 own MLP** → 07 RealMLP → … → 11 stack

| | |
|---|---|
| **Goal** | Build a neural network for tabular data from scratch, understand which ingredients matter, and check whether it adds anything to the tree models. |
| **Input** | `data/processed/X_final_*.parquet`, `y_train.parquet` |
| **Output** | `preds/oof_mlp.npy`, `preds/test_mlp.npy` (logits) |
| **Runtime** | ~1.5 min per training run on the Mac CPU; the full 5-fold × 3-seed run ≈ 25–30 min (cached per fold in `preds/mlp_folds/`) |
| **Result** | OOF AUC **0.96043**; blended with the trees it gains **+0.00013**; added to the stack of trees +0.00010 (README, method map) |

**Why a hand-built MLP at all?** Trees and networks make different mistakes, so a good network is valuable for the stack. Building it step by step — one change at a time, each judged on the same fold — shows which ingredients matter. (Spoiler: preprocessing and the learning-rate schedule matter; regularisation and width do not.)

**Stages (each stage ends with a decision):**

| Stage | Sections | What was tested | Outcome |
|---|---|---|---|
| 1. Input preprocessing | 1–5 | robust scaling + smooth clipping, log of heavy tails, zero flags, one-hot categoricals, route index | 54 numeric inputs, checked for NaN, range and train/val similarity |
| 2. Architecture + sanity tests | 6–10 | 3×256 SELU MLP; memorisation test; logistic-regression floor | memorises 2k rows (AUC 1.0); logistic regression on the same inputs 0.95734 (the floor to beat) |
| 3. Learning rate / schedule | 11–14 | lr ∈ {5e-4 … 4e-3}, β2, cyclic schedule | **lr 2e-3, cosine, β2 0.999**; nothing else mattered. Route embedding dropped: it overfits (peaks early, then drifts down) |
| 3b. Full 5-fold + blend with trees | 15–17 | 3 seeds × 5 folds | **OOF 0.96043**, Spearman 0.97–0.98 with the trees, real but small blend gain |
| 4. Regularisation and capacity | 18–20 | dropout, weight decay, label smoothing, width, depth, SiLU | nothing beat the baseline by ≥ 0.0002 |
| 4c. Input representation | 21–24 | ratings as one-hot; piecewise-linear encoding | best +0.0001 → below the threshold; baseline stays |

> **Glossary:** *Robust scaling* = (x − median)/IQR, insensitive to outliers. *Smooth clipping* = squeeze values into (−3, 3) without hard cut-offs. *Embedding* = learnable vector per category. *Last-epoch AUC* = the score after the final epoch (honest) vs *best-epoch AUC* (picked on validation, optimistic). *β2* = Adam's second-moment decay. *PLE* = piecewise-linear encoding of a number.

---

## Stage 1 — input preprocessing (variant C: robust scaling + smooth clipping)
A neural network is far more sensitive to the *scale and shape* of its inputs than a tree. A tree only asks "is x > threshold?",
so any monotone rescaling is irrelevant for it. A network multiplies inputs by weights, so a column ranging 0..489 (delay) next to a column
ranging 0..1 (probability) makes training unstable or lets the big column dominate. Preprocessing must therefore (a) put all columns on a similar scale,
(b) tame heavy tails and outliers, (c) never use validation data to fit anything (otherwise leakage).

**What goes where (decided from the real data, see section 2):**

| Group | Columns | Treatment | Why |
|---|---|---|---|
| Heavy-tailed counts / delays | `departure_delay_in_minutes`, `flight_distance_freq`, `flight_distance_type_of_travel_freq`, `fdm_departure_delay_in_minutes` | `log1p`, then robust scale + smooth clip | delay: 91% are 0 and max is 489; freq: 1 .. 11,557. Without log the tail dominates. |
| Continuous | `age`, `flight_distance`, `orig_logit`, `te_flight_distance`, all other `fdm_*` | robust scale + smooth clip | `(x - median) / IQR`, then `x / sqrt(1 + (x/3)^2)` squeezes values smoothly into (-3, 3) |
| Ratings 0..5 (13 columns) | the 13 service ratings | robust scale + smooth clip; plus a **zero flag** for the 4 columns with >= 1% zeros | 0 means "did not use the service" (not "worst grade"), so it needs its own indicator |
| Binary | `arrival_delay_missing` | -1 / +1 | symmetric around 0 |
| Categorical (4) | gender, customer_type, type_of_travel, class | one-hot | <= 3 values each; embeddings are pointless here |
| Route id | `flight_distance` | integer index for an **embedding** (routes with >= 20 rows, the rest -> index 0) | 3,521 distinct values; 2,690 routes cover 99.4% of rows |
| Dropped | `orig_prob` | not used | monotone duplicate of `orig_logit`; the logit has the better shape for a network |

Everything is **fitted on the training part of a fold only** (median, IQR, mean/std, route list) and then applied to validation and test.
`te_flight_distance` is created in-fold before the preprocessor sees the data, exactly as for the tree models.

Variant B (quantile transform) is included for the one-fold comparison in stage 2 (`SCALING = 'quantile'`).

## 1. Setup

In [29]:
# IMPORTANT (this Mac): torch must be imported BEFORE numpy/pandas/sklearn, otherwise the kernel crashes with a segmentation fault.
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
import torch; torch.set_num_threads(1)

import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, QuantileTransformer
import warnings; warnings.filterwarnings('ignore')

N_FOLDS, RANDOM_STATE = 5, 42
CANDIDATES = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDIDATES if (p / 'X_final_train.parquet').exists())

X      = pd.read_parquet(PROC_DIR / 'X_final_train.parquet')
X_test = pd.read_parquet(PROC_DIR / 'X_final_test.parquet')
y      = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int)
print('data from', PROC_DIR.resolve(), '|', X.shape, X_test.shape)

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(X, y))

def add_te_flight_distance(X_fit, y_fit, other_frames, smooth='auto'):
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=RANDOM_STATE)
    key = lambda d: d[['flight_distance']].astype(str)
    X_fit = X_fit.copy(); X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]
    out = []
    for f in other_frames:
        f = f.copy(); f['te_flight_distance'] = enc.transform(key(f))[:, 0]; out.append(f)
    return X_fit, out

data from /Users/deonissx/Documents/Hackathons/kaggle-competitions/predicting_airline_satisfaction_s6e10/data/processed | (699635, 45) (299844, 45)


## 2. Look at the raw columns first
These numbers are the reason for the choices in the table above (heavy tails, zero shares, number of routes).

In [30]:
full = pd.concat([X, X_test], ignore_index=True)
vc = full['flight_distance'].value_counts()
print('distinct routes:', len(vc), '| routes with >=20 rows:', (vc >= 20).sum(),
      f"| rows covered: {vc[vc >= 20].sum() / len(full):.1%}")
RATING = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location',
          'food_and_drink','online_boarding','seat_comfort','inflight_entertainment','on-board_service',
          'leg_room_service','baggage_handling','checkin_service','cleanliness']
zero_share = pd.Series({c: (X[c] == 0).mean() for c in RATING})
print('\nshare of zeros per rating column:'); print((zero_share * 100).round(1).to_string())
for c in ['departure_delay_in_minutes', 'flight_distance_freq', 'age']:
    print(f'\n{c}: ', X[c].quantile([0, .25, .5, .75, .99, 1]).round(1).tolist(), '(min, q25, median, q75, q99, max)')

distinct routes: 3521 | routes with >=20 rows: 2690 | rows covered: 99.4%

share of zeros per rating column:
inflight_wifi_service                1.9
departure/arrival_time_convenient    3.7
ease_of_online_booking               2.3
gate_location                        0.2
food_and_drink                       0.1
online_boarding                      1.3
seat_comfort                         0.1
inflight_entertainment               0.1
on-board_service                     0.1
leg_room_service                     0.2
baggage_handling                     0.0
checkin_service                      0.1
cleanliness                          0.1

departure_delay_in_minutes:  [0.0, 0.0, 0.0, 0.0, 30.0, 489.0] (min, q25, median, q75, q99, max)

flight_distance_freq:  [1.0, 405.0, 1482.0, 3708.0, 11557.0, 11557.0] (min, q25, median, q75, q99, max)

age:  [7.0, 27.0, 40.0, 50.0, 68.0, 85.0] (min, q25, median, q75, q99, max)


## 3. The preprocessor
`fit(X_train_part)` learns the parameters, `transform(X)` applies them and returns
- `x_num`: float32 matrix (scaled numeric columns + zero flags + binary + one-hot),
- `route_idx`: int64 vector for the route embedding (0 = rare/unknown route),
- `names`: column names of `x_num`.

In [31]:
CAT_COLS   = ['gender', 'customer_type', 'type_of_travel', 'class']
BINARY     = ['arrival_delay_missing']
LOG_COLS   = ['departure_delay_in_minutes', 'flight_distance_freq',
              'flight_distance_type_of_travel_freq', 'fdm_departure_delay_in_minutes']
DROP       = ['orig_prob']
ZERO_FLAG_MIN_SHARE = 0.01     # a zero flag for ratings where >= 1% of rows are 0
ROUTE_MIN_COUNT     = 20       # routes with fewer rows share embedding index 0
SMOOTH_CLIP = 3.0

def smooth_clip(x, c=SMOOTH_CLIP):
    return x / np.sqrt(1.0 + (x / c) ** 2)           # monotone, values squeezed into (-c, c)

# Everything that is FITTED (median, IQR, category levels, route list) is fitted on the training part of a fold only, then applied to validation/test: no leakage.
class MLPPreprocessor:
    def __init__(self, scaling='robust_clip'):
        assert scaling in ('robust_clip', 'quantile')
        self.scaling = scaling

    def _num_frame(self, X):
        N = X.drop(columns=CAT_COLS + BINARY + DROP + ['flight_distance'], errors='ignore').copy()
        N['flight_distance'] = X['flight_distance']
        for c in LOG_COLS: N[c] = np.log1p(N[c])
        return N

    def fit(self, X_tr, route_counts):
        N = self._num_frame(X_tr)
        self.num_cols = list(N.columns)
        if self.scaling == 'robust_clip':
            self.center = N.median()
            iqr = N.quantile(0.75) - N.quantile(0.25)
            self.fallback = iqr == 0                          # e.g. delay after log1p: >75% zeros
            self.scale = iqr.where(~self.fallback, N.std())   # fallback: ordinary z-score
            self.center = self.center.where(~self.fallback, N.mean())
        else:
            self.qt = QuantileTransformer(n_quantiles=1000, output_distribution='normal',
                                          subsample=200_000, random_state=RANDOM_STATE).fit(N)
        self.zero_cols = [c for c in RATING if (X_tr[c] == 0).mean() >= ZERO_FLAG_MIN_SHARE]
        self.cat_levels = {c: sorted(X_tr[c].astype(str).unique()) for c in CAT_COLS}
        keep = route_counts[route_counts >= ROUTE_MIN_COUNT].index.sort_values()
        self.route_map = {v: i + 1 for i, v in enumerate(keep)}      # 0 = rare / unknown
        self.n_routes = len(self.route_map) + 1
        return self

    def transform(self, X):
        N = self._num_frame(X)[self.num_cols]
        if self.scaling == 'robust_clip':
            A = smooth_clip(((N - self.center) / self.scale).values)
        else:
            A = self.qt.transform(N)
        parts, names = [A], list(self.num_cols)
        for c in self.zero_cols:
            parts.append((X[c] == 0).astype(float).values[:, None]); names.append(f'zero_{c}')
        for c in BINARY:
            parts.append((X[c].astype(float).values[:, None] * 2 - 1)); names.append(c)
        for c in CAT_COLS:
            for lv in self.cat_levels[c]:
                parts.append((X[c].astype(str) == lv).astype(float).values[:, None]); names.append(f'{c}={lv}')
        x_num = np.hstack(parts).astype(np.float32)
        route_idx = X['flight_distance'].map(self.route_map).fillna(0).astype(np.int64).values
        return x_num, route_idx, names

## 4. Sanity checks on fold 1
What must hold: no NaN/inf; with robust+clip every scaled value lies inside (-3, 3); train and validation statistics look alike (no sign of leakage or drift);
the fallback columns (IQR = 0) are listed; the number of embedding rows is sensible.

In [32]:
tr, va = FOLDS[0]
X_tr, X_va = X.iloc[tr], X.iloc[va]
X_tr, [X_va, X_te] = add_te_flight_distance(X_tr, y.iloc[tr], [X_va, X_test])
route_counts = pd.concat([X, X_test])['flight_distance'].value_counts()   # label-free counts, same as the freq features

SCALING = 'robust_clip'          # 'quantile' = variant B
pre = MLPPreprocessor(SCALING).fit(X_tr, route_counts)
A_tr, r_tr, names = pre.transform(X_tr)
A_va, r_va, _     = pre.transform(X_va)
A_te, r_te, _     = pre.transform(X_te)

print('x_num shapes  train/val/test:', A_tr.shape, A_va.shape, A_te.shape, '| features:', len(names))
print('NaN/inf:', int(np.isnan(A_tr).sum()), int(np.isinf(A_tr).sum()), '(must be 0, 0)')
print('route embedding rows:', pre.n_routes, '| share of val rows with rare/unknown route:', round(float((r_va == 0).mean()), 4))
if SCALING == 'robust_clip':
    print('columns with IQR = 0 (z-score fallback):', list(pre.fallback[pre.fallback].index))
    print('max |value| train / val:', round(float(np.abs(A_tr[:, :len(pre.num_cols)]).max()), 3),
          '/', round(float(np.abs(A_va[:, :len(pre.num_cols)]).max()), 3), '(must be < 3)')
print('zero flags created for:', pre.zero_cols)

n_num = len(pre.num_cols)
summary = pd.DataFrame({'train_mean': A_tr[:, :n_num].mean(0), 'val_mean': A_va[:, :n_num].mean(0),
                        'train_std': A_tr[:, :n_num].std(0), 'val_std': A_va[:, :n_num].std(0),
                        'train_min': A_tr[:, :n_num].min(0), 'train_max': A_tr[:, :n_num].max(0)}, index=pre.num_cols).round(3)
print(); print(summary.to_string())

x_num shapes  train/val/test: (559708, 54) (139927, 54) (299844, 54) | features: 54
NaN/inf: 0 0 (must be 0, 0)
route embedding rows: 2691 | share of val rows with rare/unknown route: 0.0057
columns with IQR = 0 (z-score fallback): ['departure_delay_in_minutes']
max |value| train / val: 2.999 / 2.999 (must be < 3)
zero flags created for: ['inflight_wifi_service', 'departure/arrival_time_convenient', 'ease_of_online_booking', 'online_boarding']

                                       train_mean  val_mean  train_std  val_std  train_min  train_max
age                                        -0.041    -0.040      0.575    0.574     -1.294      1.639
inflight_wifi_service                      -0.105    -0.105      0.611    0.610     -1.342      0.949
departure/arrival_time_convenient           0.079     0.076      0.699    0.699     -1.342      0.949
ease_of_online_booking                     -0.088    -0.089      0.633    0.633     -1.342      0.949
gate_location                            

## 5. How to read the table above
* **mean near 0 and std around 0.5–1** for most columns: a healthy network input. Columns with a large std (> 1.5) or a mean far from 0 deserve a comment, not silent acceptance.
* **train and val values almost identical**: the preprocessor generalises. A noticeable gap would mean that something was fitted on the wrong data.
* `departure_delay_in_minutes`: the z-score fallback is used (IQR = 0), so most rows sit at one low value and a few reach high values; that is expected.

Next stage (2): the architecture, plus a minimal training loop so that variant C and variant B can be compared on one fold.

---
# Stage 2 — architecture and a minimal training loop

**Goal of this stage:** a first working network, plus two sanity tests, plus a one-fold comparison of preprocessing variant C (robust + smooth clip) against variant B (quantile transform) and of "with / without route embedding".
Optimizer, learning-rate schedule and regularization values used here are **placeholders** (marked `PLACEHOLDER`) — they are chosen properly in stages 3 and 4.

## The network (variant A: plain MLP + route embedding)
```
route id ──► Embedding(2691 x 8) ─┐
                                   ├─ concat ─► [Linear 256 → SELU → Dropout 0.15] x 3 ─► Linear 1 ─► logit
numeric block (54 columns) ───────┘
```
* **Route embedding.** The route (`flight_distance`) is a 2,690-value identifier. An embedding gives each route its own learnable 8-number vector, i.e. the network learns a "route profile" itself,
  from the labels, in addition to the label-free profile columns (`fdm_*`) and the target encoding that are already in the input. Rare routes share index 0.
* **3 hidden layers x 256, dropout 0.15, SELU:** the base configuration of RealMLP (Holzmüller et al., 2024). Width/depth matter less than preprocessing and the LR schedule.
* **SELU + LeCun-normal initialisation:** SELU keeps activations near zero mean / unit variance only if weights are initialised that way (`kaiming_normal_` with `nonlinearity='linear'`).
* **One output logit + `BCEWithLogitsLoss`:** numerically more stable than sigmoid followed by log-loss; AUC only needs the ranking, and the logit is also what the stack will use.

## 6. Imports and device

In [33]:
import time, torch.nn as nn
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'     # 'mps' is skipped on purpose: it segfaulted in this environment
print('torch', torch.__version__, '| device:', DEVICE)
SMOKE = False           # True: 60k training rows and few epochs, only checks that the code runs
RUN_STAGE2_COMPARE = False   # section 9 (3 runs, ~3 min): already done, set True to repeat
RUN_STAGE4 = False           # sections 18-19 (stage 4 screening, ~30 min): done, nothing helped
RUN_STAGE3 = False           # sections 12-13 (14 runs, ~25 min): already done, set True to repeat

torch 2.14.1 | device: cpu


## 7. Model and training loop

In [34]:
class TabMLP(nn.Module):
    def __init__(self, d_num, n_routes, use_route=True, emb_dim=8, hidden=256, n_layers=3, dropout=0.15, act='selu'):
        super().__init__()
        self.emb = nn.Embedding(n_routes, emb_dim) if use_route else None
        d = d_num + (emb_dim if use_route else 0)
        Act = {'selu': nn.SELU, 'relu': nn.ReLU, 'silu': nn.SiLU}[act]
        layers = []
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), Act(), nn.Dropout(dropout)]
            d = hidden
        self.body, self.head = nn.Sequential(*layers), nn.Linear(d, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='linear' if act == 'selu' else 'relu')
                nn.init.zeros_(m.bias)
        if self.emb is not None:
            nn.init.normal_(self.emb.weight, 0.0, 0.1)

    def forward(self, x_num, route):
        h = x_num if self.emb is None else torch.cat([x_num, self.emb(route)], dim=1)
        return self.head(self.body(h)).squeeze(-1)

@torch.no_grad()
def predict_logits(model, A, r, bs=8192):
    model.eval(); out = []
    for i in range(0, len(A), bs):
        a = torch.from_numpy(A[i:i+bs]).to(DEVICE); rr = torch.from_numpy(r[i:i+bs]).to(DEVICE)
        out.append(model(a, rr).float().cpu().numpy())
    return np.concatenate(out)

# Learning-rate shape over training. Cosine = one smooth decay from lr to 0; at the end the weights are "settled", which is why we judge a run by its LAST epoch, not by the best epoch (the best epoch is picked on the validation set, so it is optimistic).
def lr_factor(schedule, total):
    """Multiplier of the base learning rate as a function of the step (t = step / total steps in 0..1)."""
    if schedule == 'cosine':        # one smooth decay from lr to 0
        return lambda s: 0.5 * (1 + np.cos(np.pi * min(s / total, 1.0)))
    if schedule == 'coslog4':       # RealMLP-style: 4 cycles of growing length, ends at 0 (implemented from the paper's description)
        return lambda s: 0.5 * (1 - np.cos(2 * np.pi * np.log2(1 + 15 * min(s / total, 1.0))))
    raise ValueError(schedule)

def train_model(Atr, rtr, ytr, Ava, rva, yva, n_routes, *, epochs, seed=0, batch=1024, Ate=None, rte=None, label_smooth=0.0,
                lr=2e-3, wd=0.02,                       # lr: stage 3; wd / dropout: stage 4
                betas=(0.9, 0.999), schedule='cosine',  # stage 3 knobs
                verbose=True, **model_kw):
    torch.manual_seed(seed); np.random.seed(seed)
    model = TabMLP(Atr.shape[1], n_routes, **model_kw).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, betas=betas)
    steps = int(np.ceil(len(Atr) / batch)) * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor(schedule, steps))
    A_t, r_t = torch.from_numpy(Atr).to(DEVICE), torch.from_numpy(rtr).to(DEVICE)
    y_t = torch.from_numpy(np.asarray(ytr, dtype=np.float32)).to(DEVICE)
    if label_smooth > 0: y_t = y_t * (1 - label_smooth) + 0.5 * label_smooth      # targets 0/1 -> s/2 and 1-s/2
    loss_fn = nn.BCEWithLogitsLoss()
    # Best-epoch AUC is tracked only for information; every decision in this notebook uses the last-epoch AUC.
    best = {'auc': -1.0, 'epoch': -1, 'logits': None}; hist = []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(len(Atr), device=DEVICE); run = 0.0
        for i in range(0, len(Atr), batch):
            idx = perm[i:i + batch]
            loss = loss_fn(model(A_t[idx], r_t[idx]), y_t[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            run += loss.item() * len(idx)
        lg = predict_logits(model, Ava, rva)
        auc = roc_auc_score(yva, lg); hist.append((ep, run / len(Atr), auc))
        if auc > best['auc']: best = {'auc': auc, 'epoch': ep, 'logits': lg}
        if verbose: print(f'  epoch {ep:2d} | train loss {run/len(Atr):.4f} | val AUC {auc:.5f} | {time.time()-t0:5.0f}s', flush=True)
    best['hist'] = hist
    best['last_logits'] = lg                                   # logits of the FINAL epoch (what we use for OOF)
    if Ate is not None: best['test_logits'] = predict_logits(model, Ate, rte)
    return best

## 8. Two sanity tests (do these before trusting any number)
1. **Can the network memorise a tiny set?** 2,000 rows, no dropout, no weight decay: the AUC on those same rows must approach 1.0. If it does not, there is a bug in the model or the loop.
2. **A linear baseline (logistic regression) on exactly the same inputs.** This is the floor: the network must beat it clearly. If it does not, the network is not using its extra capacity
   (learning rate, initialisation or preprocessing problem). It also verifies that the preprocessed inputs carry the signal at all.

*(A "shuffled labels -> AUC 0.5" test is deliberately not used: a network trained on random labels is a random function of informative features,
and such a function can have an AUC far from 0.5 against the true labels without any leakage, so the test would give false alarms.)*

In [35]:
from sklearn.linear_model import LogisticRegression
pre_c = MLPPreprocessor('robust_clip').fit(X_tr, route_counts)
A_tr, r_tr, names = pre_c.transform(X_tr); A_va, r_va, _ = pre_c.transform(X_va)
y_tr_np, y_va_np = y.iloc[tr].values, y.iloc[va].values

# test 1: memorise 2,000 rows
sub = slice(0, 2000)
b = train_model(A_tr[sub], r_tr[sub], y_tr_np[sub], A_tr[sub], r_tr[sub], y_tr_np[sub], pre_c.n_routes,
                epochs=150, batch=256, wd=0.0, dropout=0.0, verbose=False)
print(f'test 1 (memorise 2k rows): AUC on the same rows = {b["auc"]:.4f}  -> expect > 0.99')

# test 2: logistic regression on the same numeric block (no route embedding)
n_lr = 60000 if SMOKE else len(A_tr)
lr_model = LogisticRegression(C=1.0, max_iter=300).fit(A_tr[:n_lr], y_tr_np[:n_lr])
auc_lr = roc_auc_score(y_va_np, lr_model.decision_function(A_va))
print(f'test 2 (logistic regression, same inputs): validation AUC = {auc_lr:.5f}  -> the MLP must clearly beat this')

test 1 (memorise 2k rows): AUC on the same rows = 1.0000  -> expect > 0.99
test 2 (logistic regression, same inputs): validation AUC = 0.95734  -> the MLP must clearly beat this


## 9. One-fold comparison (fold 1, one seed)
Runs: **C + route** (our choice), **B + route** (quantile transform), **C without route embedding**.
Reference on the same fold: tuned LightGBM scored **0.96095**; a plain MLP is expected to be somewhat lower (roughly 0.955–0.959).
**How to read it:** a difference between two runs below ~0.0005 is not conclusive for a neural network with a single seed (seed noise is larger than for trees). Larger gaps are informative.

In [36]:
if RUN_STAGE2_COMPARE:
    EPOCHS = 2 if SMOKE else 15       # PLACEHOLDER
    if SMOKE:
        keep = np.random.RandomState(1).choice(len(X_tr), min(60000, len(X_tr)), replace=False)
        Xs, ys = X_tr.iloc[keep], y.iloc[tr].values[keep]
    else:
        Xs, ys = X_tr, y.iloc[tr].values

    runs = [('C + route emb', 'robust_clip', True), ('B + route emb', 'quantile', True), ('C, no route emb', 'robust_clip', False)]
    rows = []
    for label, scaling, use_route in runs:
        print(f'\n=== {label} ===')
        pre = MLPPreprocessor(scaling).fit(Xs, route_counts)
        Atr_, rtr_, _ = pre.transform(Xs); Ava_, rva_, _ = pre.transform(X_va)
        res = train_model(Atr_, rtr_, ys, Ava_, rva_, y_va_np, pre.n_routes, epochs=EPOCHS, use_route=use_route)
        rows.append({'run': label, 'best val AUC': round(res['auc'], 5), 'best epoch': res['epoch'], 'last epoch AUC': round(res['hist'][-1][2], 5)})
    print(); print(pd.DataFrame(rows).to_string(index=False))
    print(f'logistic regression (floor) = {auc_lr:.5f} | reference: tuned LightGBM on this fold = 0.96095')

## 10. What we decide after this
* Variant C vs B: keep the better one if the gap is clear (> 0.0005); otherwise keep C (simpler, no fitted quantile map).
* Route embedding on/off: keep it if it helps; if it does not, drop it (fewer parameters, no rare-route issue).
* If the best AUC is far below ~0.95, something is wrong: look at the training curves above (loss not falling → learning rate / init; train loss falling but val AUC flat → overfitting).
* Then stage 3: optimizer and learning-rate schedule.

---
# Stage 3 — learning rate, optimizer and schedule

**What stage 2 showed (fold 1):** without the route embedding the network reaches 0.95999 and is still improving at epoch 15, i.e. it is under-trained; the embedding variants peak early and then drift down (overfitting).
So from now on: **no route embedding**, and we judge a run by its **last epoch**, not the best epoch (the best epoch is picked on the validation set, so it is optimistic; the schedule decays to 0, so the last epoch is the honest number).

**Experiment A — learning rate.** Base LR in {5e-4, 1e-3, 2e-3, 4e-3}, 30 epochs, cosine decay, 2 seeds each. Too low an LR → under-trained; too high → noisy, worse final point.
**Experiment B — optimizer / schedule** at the best LR from A: AdamW β2 = 0.999 vs 0.95, and cosine vs the multi-cycle "coslog4" schedule (RealMLP). One change at a time is compared with the baseline (cosine, β2 = 0.999).

**Reading rule:** with 2 seeds, a mean difference below ~0.0005 is not conclusive → prefer the simpler setting (cosine, β2 = 0.999).

## 11. Helper: run a configuration on several seeds

In [37]:
if SMOKE:
    keep3 = np.random.RandomState(1).choice(len(A_tr), min(60000, len(A_tr)), replace=False)
    A3, r3, y3 = A_tr[keep3], r_tr[keep3], y_tr_np[keep3]
    EP3, SEEDS = 2, [0]
else:
    A3, r3, y3 = A_tr, r_tr, y_tr_np
    EP3, SEEDS = 30, [0, 1]

def run_cfg(label, **kw):
    t0 = time.time(); last, best_ep, best = [], [], []
    for s in SEEDS:
        res = train_model(A3, r3, y3, A_va, r_va, y_va_np, pre_c.n_routes, epochs=EP3, seed=s,
                          use_route=False, verbose=False, **kw)
        last.append(res['hist'][-1][2]); best.append(res['auc']); best_ep.append(res['epoch'])
    row = {'config': label, 'last AUC mean': round(float(np.mean(last)), 5), 'last AUC std': round(float(np.std(last)), 5),
           'best-epoch AUC mean': round(float(np.mean(best)), 5), 'best epochs': best_ep, 'sec/run': round((time.time() - t0) / len(SEEDS))}
    print(row, flush=True)
    return row

## 12. Experiment A — learning rate (cosine, β2 = 0.999)

In [38]:
if RUN_STAGE3:
    LRS = [5e-4, 1e-3, 2e-3, 4e-3]
    rows_a = [run_cfg(f'lr={lr:g}', lr=lr) for lr in LRS]
    tab_a = pd.DataFrame(rows_a); print(); print(tab_a.to_string(index=False))
    best_lr = LRS[int(tab_a['last AUC mean'].values.argmax())]
    print(f'\nbest LR by last-epoch AUC: {best_lr:g}   (logistic regression floor {auc_lr:.5f}, LightGBM on this fold 0.96095)')
else:
    best_lr = 2e-3


## 13. Experiment B — β2 and schedule at the best LR
* **β2 = 0.95** (instead of 0.999): the second-moment average forgets faster, which usually suits noisy tabular gradients (RealMLP default).
* **coslog4**: the LR rises and falls four times with growing cycle length and ends at 0 — a cheap form of restarts. Implemented from the paper's description, so its exact shape is our reading of it.

In [39]:
if RUN_STAGE3:
    rows_b = [dict(rows_a[LRS.index(best_lr)], config=f'lr={best_lr:g} cosine b2=0.999 (baseline, from A)')]
    rows_b.append(run_cfg(f'lr={best_lr:g} cosine b2=0.95', lr=best_lr, betas=(0.9, 0.95)))
    rows_b.append(run_cfg(f'lr={best_lr:g} coslog4 b2=0.999', lr=best_lr, schedule='coslog4'))
    rows_b.append(run_cfg(f'lr={best_lr:g} coslog4 b2=0.95', lr=best_lr, betas=(0.9, 0.95), schedule='coslog4'))
    tab_b = pd.DataFrame(rows_b); print(); print(tab_b.to_string(index=False))
    base = tab_b['last AUC mean'].iloc[0]
    tab_b['diff vs baseline'] = (tab_b['last AUC mean'] - base).round(5)
    print(); print(tab_b[['config', 'last AUC mean', 'diff vs baseline']].to_string(index=False))

## 14. Decision rule
* Pick the LR with the best last-epoch mean; if two neighbours are within 0.0005, take the smaller (more stable).
* Switch to β2 = 0.95 or coslog4 only if it gains more than ~0.0005 on the mean over 2 seeds; otherwise keep cosine and β2 = 0.999.
* If even the best run is clearly below ~0.9605 on this fold, stage 4 (regularization) and more epochs are the next levers; if the best LR is at the edge of the grid (5e-4 or 4e-3), the grid has to be extended.

**Outcome:** lr = 2e-3 with cosine decay and β2 = 0.999 stayed; β2 = 0.95 and the cyclic schedule did not gain more than the 0.0005 bar.

---
# Stage 3b — does the MLP add anything to the trees?

Before polishing the MLP further (regularization, numerical embeddings) we check what it is worth **as an ensemble member**:
1. Train the current configuration (no route embedding, lr 2e-3, cosine, 30 epochs, 3 seeds) on all 5 folds. Per fold: target encoding, preprocessing and scaler are fitted on the fold-train part only; the test set is predicted by every model.
2. The out-of-fold (OOF) logits are the seed average of the **last-epoch** predictions. Test logits are the average over all 15 models.
3. Compare with the trees: rank correlation of OOF predictions, and the AUC of simple blends.

**How to read it:** the trees are correlated 0.9994–0.9998 among themselves, so blending them gains ~+0.0001. If the MLP is clearly less correlated (≈0.99 or below) and a blend gains > +0.0003, the MLP branch is worth developing (stage 4). If the blend gains ≈ +0.0001, the MLP does not add diversity and we stop polishing it.

Resumable: every finished fold is saved to `preds/`, so an interrupted run continues from the last finished fold. Runtime ≈ 5 folds x 3 seeds x ~2 min ≈ 30 min on one CPU thread.

## 15. 5-fold training with 3 seeds

In [40]:
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working')] if p.exists()), Path('preds'))
PREDS_DIR.mkdir(exist_ok=True)
CKPT = PREDS_DIR / 'mlp_folds'; CKPT.mkdir(exist_ok=True)
# Final configuration after stages 3-4c: no route embedding, lr 2e-3, cosine schedule, beta2 0.999.
MLP_CFG = dict(lr=2e-3, schedule='cosine', betas=(0.9, 0.999), use_route=False)
FULL_EPOCHS, FULL_SEEDS = (2, [0]) if SMOKE else (30, [0, 1, 2])
fold_ids = [0] if SMOKE else list(range(N_FOLDS))

oof_mlp = np.full(len(X), np.nan); test_parts = []
for k in fold_ids:
    f_oof, f_test = CKPT / f'oof_fold{k}.npy', CKPT / f'test_fold{k}.npy'
    tr_k, va_k = FOLDS[k]
    if f_oof.exists() and f_test.exists() and not SMOKE:
        oof_mlp[va_k] = np.load(f_oof); test_parts.append(np.load(f_test)); print(f'fold {k}: loaded from disk'); continue
    Xtr_k, [Xva_k, Xte_k] = add_te_flight_distance(X.iloc[tr_k], y.iloc[tr_k], [X.iloc[va_k], X_test])
    # Fresh preprocessor per fold, fitted on that fold's training part (after in-fold target encoding).
    pre_k = MLPPreprocessor('robust_clip').fit(Xtr_k, route_counts)
    Atr_k, rtr_k, _ = pre_k.transform(Xtr_k); Ava_k, rva_k, _ = pre_k.transform(Xva_k); Ate_k, rte_k, _ = pre_k.transform(Xte_k)
    ytr_k, yva_k = y.iloc[tr_k].values, y.iloc[va_k].values
    if SMOKE:
        sm = np.random.RandomState(1).choice(len(Atr_k), min(60000, len(Atr_k)), replace=False)
        Atr_k, rtr_k, ytr_k = Atr_k[sm], rtr_k[sm], ytr_k[sm]
    v_logits, t_logits, singles = [], [], []
    # 3 seeds per fold, averaged on the logit scale: neural nets have more seed noise than trees.
    for s in FULL_SEEDS:
        res = train_model(Atr_k, rtr_k, ytr_k, Ava_k, rva_k, yva_k, pre_k.n_routes, epochs=FULL_EPOCHS, seed=s,
                          Ate=Ate_k, rte=rte_k, verbose=False, **MLP_CFG)
        v_logits.append(res['last_logits']); t_logits.append(res['test_logits']); singles.append(roc_auc_score(yva_k, res['last_logits']))
    v_avg, t_avg = np.mean(v_logits, axis=0), np.mean(t_logits, axis=0)
    oof_mlp[va_k] = v_avg; test_parts.append(t_avg)
    if not SMOKE: np.save(f_oof, v_avg); np.save(f_test, t_avg)
    print(f'fold {k}: single-seed AUC {np.round(singles, 5).tolist()} | seed-average AUC {roc_auc_score(yva_k, v_avg):.5f}', flush=True)

if not SMOKE:
    test_mlp = np.mean(test_parts, axis=0)
    print(f'\nOOF AUC of the MLP (all rows, last epoch, seed-averaged): {roc_auc_score(y, oof_mlp):.5f}')
    np.save(PREDS_DIR / 'oof_mlp.npy', oof_mlp); np.save(PREDS_DIR / 'test_mlp.npy', test_mlp)
    print('saved:', PREDS_DIR / 'oof_mlp.npy', PREDS_DIR / 'test_mlp.npy', '(logits)')

fold 0: loaded from disk
fold 1: loaded from disk
fold 2: loaded from disk
fold 3: loaded from disk
fold 4: loaded from disk

OOF AUC of the MLP (all rows, last epoch, seed-averaged): 0.96043
saved: preds/oof_mlp.npy preds/test_mlp.npy (logits)


## 16. Correlation with the trees and blend check
Tree predictions are read from `preds/` (CatBoost v1, XGB, and the tuned LightGBM if `oof_lgbm_tuned_final.npy` is there). Everything is converted to logits; correlations are Spearman (rank) correlations.
Blend = `w · MLP + (1-w) · mean(trees)` on logits; w is scanned on the same OOF — a one-parameter scan, so the optimism is negligible, but treat the last digit with care.

In [41]:
if SMOKE:
    print('smoke mode: skipped')
else:
    from scipy.stats import spearmanr
    def to_logit(p):
        p = np.asarray(p, dtype=float)
        return p if (p.min() < 0 or p.max() > 1) else np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    names_t = {'cat_v1': 'oof_cat_v1.npy', 'xgb': 'oof_xgb.npy', 'lgbm_tuned': 'oof_lgbm_tuned_final.npy'}
    trees = {n: to_logit(np.load(PREDS_DIR / f)) for n, f in names_t.items() if (PREDS_DIR / f).exists()}
    print('trees found:', list(trees))
    rows = [{'model': 'MLP', 'OOF AUC': round(roc_auc_score(y, oof_mlp), 5), 'spearman vs MLP': 1.0}]
    for n, p in trees.items():
        rows.append({'model': n, 'OOF AUC': round(roc_auc_score(y, p), 5), 'spearman vs MLP': round(spearmanr(p, oof_mlp)[0], 5)})
    print(pd.DataFrame(rows).to_string(index=False))
    tree_mean = np.mean(list(trees.values()), axis=0)
    print(f'\nmean of trees: AUC {roc_auc_score(y, tree_mean):.5f}')
    blend = []
    for w in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
        blend.append({'w (MLP)': w, 'OOF AUC': round(roc_auc_score(y, w * oof_mlp + (1 - w) * tree_mean), 5)})
    bt = pd.DataFrame(blend); bt['gain vs trees'] = (bt['OOF AUC'] - bt['OOF AUC'].iloc[0]).round(5)
    print(bt.to_string(index=False))
    for n, p in trees.items():                         # per-fold sign check of the gain at the best w
        pass
    wb = bt.loc[bt['OOF AUC'].idxmax(), 'w (MLP)']
    print(f'\nbest w = {wb}; per-fold gain (blend - trees) at that w:')
    print([round(roc_auc_score(y.iloc[va], wb * oof_mlp[va] + (1 - wb) * tree_mean[va]) - roc_auc_score(y.iloc[va], tree_mean[va]), 5) for _, va in FOLDS])

trees found: ['cat_v1', 'xgb']
 model  OOF AUC  spearman vs MLP
   MLP  0.96043          1.00000
cat_v1  0.96064          0.97694
   xgb  0.96066          0.97064

mean of trees: AUC 0.96077
 w (MLP)  OOF AUC  gain vs trees
     0.0  0.96077        0.00000
     0.1  0.96084        0.00007
     0.2  0.96088        0.00011
     0.3  0.96090        0.00013
     0.4  0.96090        0.00013
     0.5  0.96088        0.00011
     0.6  0.96083        0.00006

best w = 0.3; per-fold gain (blend - trees) at that w:
[2e-05, 0.0001, 0.00017, 0.00022, 0.00016]


## 17. Decision
* Per-fold gains all positive and mean > +0.0003 → the MLP is a real ensemble member; continue to stage 4 (regularization, numerical embeddings), then the final stack.
* Gain ≈ +0.0001 or mixed signs → the MLP adds the same information as the trees; stop polishing, move on to TabPFN and the final stack.

**Outcome:** the MLP is a real but small ensemble member: blending it with the mean of the trees gains **+0.00013** at w = 0.3 (OOF 0.96077 → 0.96090); added to the stack of the four tree models it gains +0.00010 (nested CV, 5 / 5 folds). Not large enough to justify unlimited polishing — but enough to test regularisation once (stage 4).

> **Later update.** Once RealMLP (`07`, `10`) and TabPFN (`08`) entered the stack, the own MLP stopped adding anything: removing it from the final 10-model stack changes the nested CV by +0.00001. It remains in the stack as a harmless member and as the documented learning path to RealMLP.

---
# Stage 4 — regularization and capacity

**What is tested** (one change at a time from the baseline: 3x256 SELU, dropout 0.15, weight decay 0.02, no label smoothing, no route embedding, lr 2e-3, cosine, 30 epochs):

| factor | variants |
|---|---|
| dropout | 0.05, 0.30 |
| weight decay | 0.1 |
| label smoothing | 0.1 (targets 0/1 -> 0.05/0.95; flattens the loss on easy rows; ranking metric, so we only check whether it helps) |
| width | 512 |
| depth | 4 layers |
| activation | SiLU instead of SELU |
| route embedding | back on, but with dropout 0.30 (it overfitted at dropout 0.15) |

**Screening:** one seed per variant on fold 1 (the seed-to-seed spread at the last epoch was only ~0.00003 in stage 3, so one seed is enough to see a difference of ~0.0002). **Reading rule:** a variant counts as an improvement if it beats the baseline run by ≥ 0.0002 at the last epoch.
Then the improvements are **combined**, and the combination is confirmed with 2 seeds on fold 1 and with 1 seed on a different fold (fold 2) — a variant that wins only on the fold it was selected on is not trusted.

## 18. Screening (one seed per variant, fold 1)

In [42]:
if RUN_STAGE4:
    BASE = dict(lr=2e-3, use_route=False)
    VARIANTS = [  # (factor, label, kwargs added to the baseline)
        ('dropout', 'dropout 0.05', dict(dropout=0.05)),
        ('dropout', 'dropout 0.30', dict(dropout=0.30)),
        ('wd',      'weight decay 0.1', dict(wd=0.1)),
        ('ls',      'label smoothing 0.1', dict(label_smooth=0.1)),
        ('width',   'width 512', dict(hidden=512)),
        ('depth',   '4 layers', dict(n_layers=4)),
        ('act',     'SiLU', dict(act='silu')),
        ('route',   'route emb + dropout 0.30', dict(use_route=True, dropout=0.30)),
    ]

    def run_one(kw, A_tr_, r_tr_, y_tr_, A_va_, r_va_, y_va_, n_routes, seed=0, epochs=None):
        t0 = time.time()
        res = train_model(A_tr_, r_tr_, y_tr_, A_va_, r_va_, y_va_, n_routes, epochs=epochs or EP3, seed=seed, verbose=False, **kw)
        return res['hist'][-1][2], round(time.time() - t0)

    screen = []
    auc_b, sec_b = run_one(BASE, A3, r3, y3, A_va, r_va, y_va_np, pre_c.n_routes)
    screen.append({'factor': '-', 'variant': 'baseline', 'last AUC': round(auc_b, 5), 'diff': 0.0, 'sec': sec_b}); print(screen[-1], flush=True)
    for factor, label, add in VARIANTS:
        a, sec = run_one({**BASE, **add}, A3, r3, y3, A_va, r_va, y_va_np, pre_c.n_routes)
        screen.append({'factor': factor, 'variant': label, 'last AUC': round(a, 5), 'diff': round(a - auc_b, 5), 'sec': sec}); print(screen[-1], flush=True)
    scr = pd.DataFrame(screen); print(); print(scr.to_string(index=False))

## 19. Combine the improvements
For each factor the best variant is taken if it beat the baseline by ≥ 0.0002 (the route variant is only reported, not auto-combined: it changes the input, not just the regularization).
The combination is then compared with the baseline using 2 seeds on fold 1 and 1 seed on fold 2.

In [43]:
if RUN_STAGE4:
    THR = 0.0002
    combo_kw, picked = dict(BASE), []
    for factor in ['dropout', 'wd', 'ls', 'width', 'depth', 'act']:
        sub_ = scr[(scr['factor'] == factor) & (scr['diff'] >= THR)]
        if len(sub_):
            best_row = sub_.sort_values('diff').iloc[-1]; picked.append(best_row['variant'])
            combo_kw.update(next(add for f, l, add in VARIANTS if l == best_row['variant']))
    print('improvements picked:', picked or 'none -> the baseline stays')
    print('combination config:', combo_kw)

    if not picked:
        print('nothing to combine')
    else:
        # fold 1, 2 seeds
        base2 = [run_one(BASE, A3, r3, y3, A_va, r_va, y_va_np, pre_c.n_routes, seed=s)[0] for s in [0, 1]]
        comb2 = [run_one(combo_kw, A3, r3, y3, A_va, r_va, y_va_np, pre_c.n_routes, seed=s)[0] for s in [0, 1]]
        print(f'fold 1 (2 seeds): baseline {np.mean(base2):.5f} | combination {np.mean(comb2):.5f} | diff {np.mean(comb2) - np.mean(base2):+.5f}')
        # fold 2, 1 seed (its own preprocessing, fitted on its own training part)
        tr2, va2 = FOLDS[1]
        X_tr2, [X_va2] = add_te_flight_distance(X.iloc[tr2], y.iloc[tr2], [X.iloc[va2]])
        pre2 = MLPPreprocessor('robust_clip').fit(X_tr2, route_counts)
        A_tr2, r_tr2, _ = pre2.transform(X_tr2); A_va2, r_va2, _ = pre2.transform(X_va2)
        y_tr2, y_va2 = y.iloc[tr2].values, y.iloc[va2].values
        if SMOKE:
            k2 = np.random.RandomState(2).choice(len(A_tr2), min(60000, len(A_tr2)), replace=False); A_tr2, r_tr2, y_tr2 = A_tr2[k2], r_tr2[k2], y_tr2[k2]
        b_f2 = run_one(BASE, A_tr2, r_tr2, y_tr2, A_va2, r_va2, y_va2, pre2.n_routes)[0]
        c_f2 = run_one(combo_kw, A_tr2, r_tr2, y_tr2, A_va2, r_va2, y_va2, pre2.n_routes)[0]
        print(f'fold 2 (1 seed):  baseline {b_f2:.5f} | combination {c_f2:.5f} | diff {c_f2 - b_f2:+.5f}')

## 20. Decision
* Combination better than the baseline on **both** folds (> +0.0002 each) → it becomes the MLP configuration for the final 5-fold run (stage 5).
* Better on fold 1 only, or by less than 0.0002 → keep the baseline (or the single factor that is confirmed).
* If **nothing** reaches +0.0002, the MLP is limited by its inputs and architecture, not by regularization: we go to numerical embeddings (stage 4c).

**Outcome:** no regularisation or capacity variant reached +0.0002, so the MLP is limited by its inputs and architecture, not by regularisation. Went on to input representation (stage 4c).

---
# Stage 4c — a better representation of the inputs

**Why:** stage 4 showed that neither regularization nor capacity changes the result. Trees cut each feature at exact thresholds; our MLP sees each feature as one number and can only bend it smoothly. Two standard ways to remove that handicap:

* **A. Ratings as categories.** The 13 ratings (0–5) are given as one-hot vectors (6 levels each) *in addition to* the scaled number. The effect of a rating on satisfaction is not necessarily monotonic (e.g. "0 = not rated" behaves differently from "1 = very bad"); one-hot lets the network learn each level separately.
* **B. Piecewise-linear encoding (PLE)** of the continuous features (age, delay, distance, frequencies, route profile `fdm_*`, `orig_logit`, target encoding). Each feature is cut into 16 quantile bins (edges fitted on the fold-train part only). A value is encoded as 16 numbers: 1 for bins entirely below the value, a fraction in the bin it falls into, 0 above. This gives the network "step-like" building blocks, as in Gorishniy et al. (2022), and is the base of the numerical-embedding variants in TabM / RealMLP.

Both are added *next to* the existing 54 scaled columns, so nothing is lost. Runs are 2–3x slower than before (more input columns).
**Screening:** baseline, A, B, A+B — one seed, fold 1, last epoch; improvement if ≥ +0.0002; the winner is confirmed on fold 2.

## 21. Feature builders (fitted on the training part only)

In [44]:
def run_one(kw, A_tr_, r_tr_, y_tr_, A_va_, r_va_, y_va_, n_routes, seed=0, epochs=None):
    t0 = time.time()
    res = train_model(A_tr_, r_tr_, y_tr_, A_va_, r_va_, y_va_, n_routes, epochs=epochs or EP3, seed=seed, verbose=False, **kw)
    return res['hist'][-1][2], round(time.time() - t0)

RATING_LEVELS = list(range(6))
PLE_COLS = [c for c in pre_c.num_cols if c not in RATING and c in X_tr.columns]
print(f'PLE on {len(PLE_COLS)} continuous columns, one-hot on {len(RATING)} ratings x {len(RATING_LEVELS)} levels')

def rating_onehot(Xd):
    R = np.rint(Xd[RATING].values).clip(0, 5).astype(np.int8)
    return np.concatenate([(R[:, [j]] == np.arange(6)[None, :]) for j in range(R.shape[1])], axis=1).astype(np.float32)

# Piecewise-linear encoding: each continuous feature is cut into 16 quantile bins; a value is encoded as a vector that is 1 for bins fully below it, a fraction inside its own bin, 0 above. Gives the network "threshold-like" inputs, like a tree has.
def ple_fit(Xd, cols, n_bins=16):
    edges = {}
    for c in cols:
        e = np.unique(np.quantile(Xd[c].values.astype(float), np.linspace(0, 1, n_bins + 1)))
        if len(e) >= 2: edges[c] = e
    return edges

def ple_transform(Xd, edges):
    out = []
    for c, e in edges.items():
        x = Xd[c].values.astype(np.float64)[:, None]
        out.append(np.clip((x - e[:-1][None, :]) / (e[1:] - e[:-1])[None, :], 0, 1).astype(np.float32))
    return np.concatenate(out, axis=1)

ple_edges = ple_fit(X_tr, PLE_COLS)
E_A_tr, E_A_va = rating_onehot(X_tr), rating_onehot(X_va)
E_B_tr, E_B_va = ple_transform(X_tr, ple_edges), ple_transform(X_va, ple_edges)
print('extra columns: A', E_A_tr.shape[1], '| B', E_B_tr.shape[1], '| PLE bins per feature (min/median/max):',
      min(len(e) - 1 for e in ple_edges.values()), int(np.median([len(e) - 1 for e in ple_edges.values()])), max(len(e) - 1 for e in ple_edges.values()))
print('PLE value range train/val:', float(E_B_tr.min()), float(E_B_tr.max()), '/', float(E_B_va.min()), float(E_B_va.max()), '(must be 0..1)')

PLE on 27 continuous columns, one-hot on 13 ratings x 6 levels
extra columns: A 78 | B 404 | PLE bins per feature (min/median/max): 2 16 16
PLE value range train/val: 0.0 1.0 / 0.0 1.0 (must be 0..1)


## 22. Screening on fold 1 (baseline, A, B, A+B)

In [45]:
def with_extra(A_base, *extras): return np.concatenate([A_base] + list(extras), axis=1)
if SMOKE:
    EA_tr3, EB_tr3 = E_A_tr[keep3], E_B_tr[keep3]
else:
    EA_tr3, EB_tr3 = E_A_tr, E_B_tr
BASE = dict(lr=2e-3, use_route=False)
sets = {'baseline': (A3, A_va), 'A: ratings one-hot': (with_extra(A3, EA_tr3), with_extra(A_va, E_A_va)),
        'B: PLE': (with_extra(A3, EB_tr3), with_extra(A_va, E_B_va)),
        'A+B': (with_extra(A3, EA_tr3, EB_tr3), with_extra(A_va, E_A_va, E_B_va))}
rows4c = []
for label, (Atr_, Ava_) in sets.items():
    a, sec = run_one(BASE, Atr_, r3, y3, Ava_, r_va, y_va_np, pre_c.n_routes)
    rows4c.append({'variant': label, 'inputs': Atr_.shape[1], 'last AUC': round(a, 5), 'sec': sec}); print(rows4c[-1], flush=True)
t4c = pd.DataFrame(rows4c); t4c['diff vs baseline'] = (t4c['last AUC'] - t4c['last AUC'].iloc[0]).round(5)
print(); print(t4c.to_string(index=False))

{'variant': 'baseline', 'inputs': 54, 'last AUC': 0.96014, 'sec': 99}
{'variant': 'A: ratings one-hot', 'inputs': 132, 'last AUC': 0.96024, 'sec': 98}
{'variant': 'B: PLE', 'inputs': 458, 'last AUC': 0.96017, 'sec': 110}
{'variant': 'A+B', 'inputs': 536, 'last AUC': 0.9601, 'sec': 117}

           variant  inputs  last AUC  sec  diff vs baseline
          baseline      54   0.96014   99           0.00000
A: ratings one-hot     132   0.96024   98           0.00010
            B: PLE     458   0.96017  110           0.00003
               A+B     536   0.96010  117          -0.00004


## 23. Confirmation on fold 2
The best variant (if it beats the baseline by ≥ 0.0002) is run on fold 2 together with the baseline; both folds must agree.

In [46]:
best_row = t4c.iloc[1:].sort_values('diff vs baseline').iloc[-1]
print('best variant:', best_row['variant'], '| diff on fold 1:', best_row['diff vs baseline'])
if best_row['diff vs baseline'] < 0.0002:
    print('no variant reaches +0.0002 -> the input representation is not the limit either; the baseline stays')
else:
    tr2, va2 = FOLDS[1]
    X_tr2, [X_va2] = add_te_flight_distance(X.iloc[tr2], y.iloc[tr2], [X.iloc[va2]])
    pre2 = MLPPreprocessor('robust_clip').fit(X_tr2, route_counts)
    A_tr2, r_tr2, _ = pre2.transform(X_tr2); A_va2, r_va2, _ = pre2.transform(X_va2)
    y_tr2, y_va2 = y.iloc[tr2].values, y.iloc[va2].values
    edges2 = ple_fit(X_tr2, PLE_COLS)
    def build(Xd, A, label):
        parts = [A]
        if 'A' in label.split(':')[0] or label == 'A+B': parts.append(rating_onehot(Xd))
        if 'B' in label.split(':')[0] or label == 'A+B': parts.append(ple_transform(Xd, edges2))
        return np.concatenate(parts, axis=1)
    if SMOKE:
        k2 = np.random.RandomState(2).choice(len(A_tr2), min(60000, len(A_tr2)), replace=False)
        X_tr2, A_tr2, r_tr2, y_tr2 = X_tr2.iloc[k2], A_tr2[k2], r_tr2[k2], y_tr2[k2]
    b2 = run_one(BASE, A_tr2, r_tr2, y_tr2, A_va2, r_va2, y_va2, pre2.n_routes)[0]
    v2 = run_one(BASE, build(X_tr2, A_tr2, best_row['variant']), r_tr2, y_tr2, build(X_va2, A_va2, best_row['variant']), r_va2, y_va2, pre2.n_routes)[0]
    print(f'fold 2: baseline {b2:.5f} | {best_row["variant"]} {v2:.5f} | diff {v2 - b2:+.5f}')

best variant: A: ratings one-hot | diff on fold 1: 0.0001
no variant reaches +0.0002 -> the input representation is not the limit either; the baseline stays


## 24. Decision
* The winner improves by ≥ 0.0002 on **both** folds → it becomes the MLP input for the final 5-fold run (stage 5).
* Only A or only B wins → keep that one. A+B wins but each alone does not → keep A+B.
* Nothing reaches +0.0002 → the MLP stays as it is (OOF 0.96043); we stop polishing it and spend the effort on the other members (TabPFN, LGBM, other ideas).

**Outcome:** best variant (ratings as one-hot) gained only +0.0001 on fold 1 (baseline 0.96014, one-hot 0.96024, PLE 0.96017, both 0.96010) → baseline kept.

## Summary & decisions
- **Final MLP:** 54 preprocessed inputs, 3×256 SELU, dropout 0.15, AdamW (lr 2e-3, wd 0.02), cosine schedule, no route embedding, 30 epochs, 3 seeds per fold → **OOF AUC 0.96043**.
- **What mattered:** input preprocessing and the learning-rate schedule. **What did not:** width, depth, dropout, weight decay, label smoothing, activation, one-hot ratings, piecewise-linear encoding.
- **Limit reached:** a plain MLP on these features ≈ 0.9604 — below tuned LightGBM (0.96088). Likely missing ingredients are exactly those of the community **RealMLP** recipe: categorical twins, PLR embeddings, very high lr with schedule, label smoothing, ensembling of many networks.
- **Next:** `07_realmlp` runs that recipe (0.96082 on the same features) and `10` adds richer features (0.96129).